# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
List all available record sets and their fields using their `@id` values.

In [ ]:
# List all record sets with their `@id` and field information
record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name')}")
    fields = rs.get('fields', [])
    if fields:
        print(f"  Fields:")
        for f in fields:
            print(f"    Field @id: {f['@id']}; Name: {f.get('name')}")
    print()

## 3. Data Extraction
Load data from available record sets into Pandas DataFrames. Use the record set and field `@id`s identified above.

In [ ]:
# Collect the @id values for each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load all available record sets as DataFrames
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nColumns in first record set ({first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. Adapt fields based on available data and their `@id`s.

In [ ]:
# Example: pick the first loaded record set and attempt EDA.
from pandas.api.types import is_numeric_dtype

# Use the first record set that contains data
record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        record_set_id = rs_id
        break

if record_set_id:
    df = dataframes[record_set_id]
    # Find a numeric column by checking dtype
    numeric_field_id = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id:
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try a group-by on another (non-numeric) column
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and not is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            print(f"\nGrouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found for analysis.")
else:
    print("No record set with data available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Proceed with visualization only if numeric and group field were found
if 'numeric_field_id' in locals() and numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Cannot generate visualization: no numeric field or filtered data.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load the FAIR^2 dataset from a Croissant schema, examined its structure via record set and field `@id`s, and performed initial exploratory data analysis and visualization. Using `@id` for referencing ensures consistent access to metadata. Further analysis and modeling can now be performed based on the dataset's fields.
